# Occlusion Sensitivity

            用遮挡验证 CAM 热区是否真的影响预测。

            比较：

            - 遮挡 CAM top-k hot region
            - 遮挡 random region
            - 遮挡 cold region

            如果 hot region 遮挡带来的 confidence/probability drop 明显更大，说明 CAM 热区具有更强判别贡献。


In [ ]:

from pathlib import Path
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

EXP_ROOT = Path("/data/zengqiang/experiments/ncfm_medmnist_ablation_20260519")

def require_exp_root():
    if not EXP_ROOT.exists():
        raise FileNotFoundError(
            f"EXP_ROOT not found: {EXP_ROOT}. "
            "Edit EXP_ROOT in the first code cell to your experiment directory."
        )

def ensure_report_dir(*parts):
    path = EXP_ROOT / "reports" / "cam" / Path(*parts)
    path.mkdir(parents=True, exist_ok=True)
    return path

def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def normalize_group(name):
    if name == "real_train":
        return "real_train"
    if name.startswith("ipc10_"):
        return name[len("ipc10_"):]
    return name

def resolve_result_path(value):
    p = Path(str(value))
    if p.exists():
        return p
    if str(value).startswith("/"):
        return p
    q = EXP_ROOT / value
    return q

def load_eval_metrics():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "runs").glob("*/ipc10/*/eval_metrics_best.json")):
        item = read_json(path)
        item["dataset"] = path.parents[2].name
        item["group"] = path.parent.name
        item["metrics_path"] = str(path)
        rows.append(item)
    if not rows:
        return pd.DataFrame()
    df = pd.DataFrame(rows)
    order = {"A_pure_ncfd_wopsi": 0, "B_minmax_ncfm_psi": 1, "C_code_default_enhanced": 2}
    df["_order"] = df["group"].map(order).fillna(99)
    return df.sort_values(["dataset", "_order"]).drop(columns=["_order"])

def load_cam_summaries():
    require_exp_root()
    rows = []
    for path in sorted((EXP_ROOT / "results" / "cam").glob("*/*/summary.csv")):
        dataset = path.parents[1].name
        group = normalize_group(path.parent.name)
        df = pd.read_csv(path)
        if df.empty:
            continue
        df["dataset"] = dataset
        df["group"] = group
        df["summary_path"] = str(path)
        rows.append(df)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    for col in ["index", "y_true", "y_pred", "confidence", "correct", "cam_entropy", "topk_activation_ratio"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

def cam_group_summary(cam_df):
    if cam_df.empty:
        return cam_df
    grouped = (
        cam_df.groupby(["dataset", "group"], as_index=False)
        .agg(
            n=("index", "count"),
            cam_acc=("correct", "mean"),
            mean_confidence=("confidence", "mean"),
            mean_entropy=("cam_entropy", "mean"),
            mean_top10_mass=("topk_activation_ratio", "mean"),
            correct_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_entropy=("cam_entropy", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
            correct_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 1].mean()),
            wrong_top10_mass=("topk_activation_ratio", lambda s: s[cam_df.loc[s.index, "correct"] == 0].mean()),
        )
    )
    order = {"real_train": 0, "A_pure_ncfd_wopsi": 1, "B_minmax_ncfm_psi": 2, "C_code_default_enhanced": 3}
    grouped["_order"] = grouped["group"].map(order).fillna(99)
    return grouped.sort_values(["dataset", "_order"]).drop(columns=["_order"])

import sys
REPO_ROOT = EXP_ROOT / "src" / "NCFM_medmnist_clean"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from torch.utils.data import DataLoader, Subset
from utils.utils import define_model
from utils.ddp import load_state_dict
from data.medmnist import get_medmnist_root, build_medmnist_dataset, get_medmnist_nclass, get_medmnist_nch, register_medmnist_stats
from data.transform import transform_medmnist
from data.dataset_statistics import MEANS, STDS


In [ ]:

def get_checkpoint(dataset, group):
    if group == "real_train":
        return EXP_ROOT / "checkpoints" / "pretrain" / dataset / "premodel0_trained.pth.tar"
    return EXP_ROOT / "checkpoints" / "synthetic_train" / dataset / f"ipc10_{group}_best.pth.tar"

def load_model(dataset, group, device="cuda"):
    nch = get_medmnist_nch(dataset)
    nclass = get_medmnist_nclass(dataset)
    model = define_model(dataset, "instance", "convnet", nch, 3, 1.0, nclass, print, 28).to(device)
    load_state_dict(str(get_checkpoint(dataset, group)), model)
    model.eval()
    return model

def load_test_dataset(dataset):
    root = get_medmnist_root(EXP_ROOT / "data")
    register_medmnist_stats(dataset, root, 28)
    _, test_transform = transform_medmnist(dataset, size=28, augment=False, normalize=True)
    return build_medmnist_dataset(dataset, root, split="test", transform=test_transform, size=28, download=True)

def read_cam_mask(path_value, mode="hot", fraction=0.10):
    path = resolve_result_path(path_value)
    img = np.asarray(Image.open(path).convert("RGB")).astype(np.float32) / 255.0
    cam = img[..., 0]
    values = cam.reshape(-1)
    k = max(1, int(round(values.size * fraction)))
    if mode == "hot":
        idx = np.argsort(values)[-k:]
    elif mode == "cold":
        idx = np.argsort(values)[:k]
    else:
        rng = np.random.default_rng(0)
        idx = rng.choice(values.size, size=k, replace=False)
    mask = np.zeros(values.size, dtype=bool)
    mask[idx] = True
    return mask.reshape(cam.shape)

def apply_mask_mean_fill(x, mask):
    # Inputs are normalized, so zero corresponds approximately to channel mean.
    out = x.clone()
    m = torch.tensor(mask, dtype=torch.bool, device=x.device)
    out[:, m] = 0.0
    return out

@torch.no_grad()
def occlusion_rows(dataset, group, max_samples=100, fraction=0.10, device="cuda"):
    cam_df = load_cam_summaries()
    sub = cam_df[(cam_df.dataset == dataset) & (cam_df.group == group)].copy().head(max_samples)
    ds = load_test_dataset(dataset)
    model = load_model(dataset, group, device=device)
    rows = []
    for _, row in sub.iterrows():
        idx = int(row["index"])
        x, y = ds[idx]
        x = x.to(device)
        y = int(y)
        logits = model(x.unsqueeze(0))
        probs = torch.softmax(logits, dim=1)[0]
        pred = int(probs.argmax().item())
        base_prob = float(probs[pred].item())
        item = {"dataset": dataset, "group": group, "index": idx, "y_true": y, "pred": pred, "base_prob": base_prob}
        for mode in ["hot", "random", "cold"]:
            mask = read_cam_mask(row["cam_path"], mode=mode, fraction=fraction)
            x_occ = apply_mask_mean_fill(x, mask)
            p_occ = torch.softmax(model(x_occ.unsqueeze(0)), dim=1)[0]
            item[f"{mode}_prob"] = float(p_occ[pred].item())
            item[f"{mode}_drop"] = base_prob - item[f"{mode}_prob"]
        rows.append(item)
    return pd.DataFrame(rows)

# Example:
# occ = occlusion_rows("pneumoniamnist", "C_code_default_enhanced", max_samples=50)
# display(occ.describe())


In [ ]:

def summarize_occlusion(occ):
    cols = ["hot_drop", "random_drop", "cold_drop"]
    out = occ.groupby(["dataset", "group"])[cols].agg(["mean", "std", "count"])
    display(out)
    ax = occ[cols].mean().plot(kind="bar", figsize=(6, 4))
    ax.set_title("Mean probability drop after occlusion")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()

# Example:
# summarize_occlusion(occ)


## 解读提示

            - `hot_drop >> random_drop/cold_drop`：CAM 热区确实更关键。
            - `hot_drop` 不明显：CAM 可能只是相关而非因果，或者模型依赖更分散的结构。
            - 建议先在每个 dataset 的 best group 跑 50-100 张样本做 sanity check。
